<a href="https://colab.research.google.com/github/hacaizacr/sp500_trend_classifier/blob/main/Grupo2_FinancialModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Grupo 2 - FM

## *Tres fuentes de datos*

In [ ]:
import pandas as pd
import requests
import io
import numpy as np
import yfinance as yf

### TABLA 1: WIKIPEDIA (Web Scraping - Metadatos de Sectores y Tickers)

In [ ]:
url_wiki = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url_wiki, headers=headers)
tablas_wiki = pd.read_html(io.StringIO(response.text))
df_wiki = tablas_wiki[0][['Symbol', 'Security', 'GICS Sector']].copy()

# Estandarización clave: Reemplazar puntos por guiones en los Tickers (BRK.B -> BRK-B)
df_wiki['Ticker'] = df_wiki['Symbol'].str.replace('.', '-', regex=False).str.strip()
df_wiki.drop(columns=['Symbol'], inplace=True)

print(f"✓ Wikipedia cargada correctamente. Total registros: {len(df_wiki)}")

✓ Wikipedia cargada correctamente. Total registros: 503


### TABLA 2: YAHOO FINANCE (Series Temporales y Precios Históricos)

In [ ]:
tickers_muestra = df_wiki['Ticker'].head(50).tolist()

# Descargar precios ajustados de cierre
datos_yf = yf.download(tickers_muestra, start="2021-01-01", end="2026-01-01")

if isinstance(datos_yf.columns, pd.MultiIndex):
    if 'Adj Close' in datos_yf.columns.levels[0]:
        df_precios = datos_yf['Adj Close'].copy()
    elif 'Close' in datos_yf.columns.levels[0]:
        df_precios = datos_yf['Close'].copy()
    else:
        df_precios = datos_yf.copy()
else:
    df_precios = datos_yf.copy()

print(f"✓ Yahoo Finance cargado correctamente. Dimensiones de matriz: {df_precios.shape}")

/tmp/ipykernel_3181/1040886481.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos_yf = yf.download(tickers_muestra, start="2021-01-01", end="2026-01-01")
[*********************100%***********************]  50 of 50 completed

✓ Yahoo Finance cargado correctamente. Dimensiones de matriz: (1255, 50)


# Tabla 3: Kaggle / Simulado de fundamentales (Formatos con $B, %, e Infinitos)

In [ ]:
np.random.seed(42)
data_kaggle = {
    'Ticker': tickers_muestra,
    'Revenue_Growth': [f"${np.random.randint(10, 500)}.{np.random.randint(1, 99)}B" if i % 5 != 0 else np.nan for i in range(len(tickers_muestra))],
    'Net_Income': [f"({np.random.randint(1, 50)}M)" if i % 4 == 0 else f"${np.random.randint(1, 80)}B" for i in range(len(tickers_muestra))],
    'Profit_Margin': [f"{round(np.random.uniform(5, 35), 2)}%" if i % 6 != 0 else "N/A" for i in range(len(tickers_muestra))],
    'Debt_to_Equity': [np.random.choice([0.45, 1.20, 3.80, np.nan, np.inf]) for _ in range(len(tickers_muestra))],
    'PE_Ratio': [np.random.choice([15.4, 28.1, 85.0, np.inf, -np.inf, "MISSING"]) for _ in range(len(tickers_muestra))]
}

df_fundamentales = pd.DataFrame(data_kaggle)
print(f"✓ Fundamentales de Kaggle cargados correctamente. Total registros: {len(df_fundamentales)}")

✓ Fundamentales de Kaggle cargados correctamente. Total registros: 50


In [ ]:
print("="*80)
print(" PASO 2: CONSTRUCCIÓN DEL TARGET Y UNIÓN DE TABLAS (pd.merge) ")
print("="*80)

# 2.1. CÁLCULO DEL TARGET A PARTIR DE PRECIOS HISTÓRICOS (YAHOO FINANCE)

# Obtener primer precio válido (2021) y último precio válido (2026) por Ticker
precio_inicial = df_precios.bfill().iloc[0]
precio_final = df_precios.ffill().iloc[-1]

# Calcular el rendimiento porcentual total en 5 años
rendimiento = (precio_final - precio_inicial) / precio_inicial

# Crear DataFrame del Target de forma limpia sin solapar índice con columnas
df_target = pd.DataFrame({
    'Ticker': rendimiento.index.astype(str),
    'Rendimiento_5Y': rendimiento.values,
    'Target': (rendimiento >= 0.50).astype(int)
}).reset_index(drop=True)

print(f"✓ Target calculado correctamente para {len(df_target)} acciones.")
print("Distribución de clases del Target (0 = No escaló, 1 = Escaló):")
print(df_target['Target'].value_counts())

# 2.2. UNIÓN DE LAS 3 FUENTES DE DATOS (INNER JOIN)

# Asegurar que df_wiki y df_fundamentales tengan Ticker como columna limpia
df_wiki_clean = df_wiki.reset_index(drop=True)
df_fund_clean = df_fundamentales.reset_index(drop=True)

# Cruce 1: Wikipedia + Target de Yahoo Finance
df_merged = pd.merge(df_wiki_clean, df_target, on='Ticker', how='inner')

# Cruce 2: Resultado + Fundamentales de Kaggle
df_raw = pd.merge(df_merged, df_fund_clean, on='Ticker', how='inner')

print("\n" + "="*80)
print(f"✓ UNIÓN COMPLETADA EXITOSAMENTE. Dimensiones finales: {df_raw.shape}")
print("Verificación de duplicados en la clave Ticker:", df_raw.duplicated(subset=['Ticker']).sum())
print("="*80)

print("\nVista previa del Dataset Unificado (Aún Sucio):")
print(df_raw.head(5))

 PASO 2: CONSTRUCCIÓN DEL TARGET Y UNIÓN DE TABLAS (pd.merge) 
✓ Target calculado correctamente para 50 acciones.
Distribución de clases del Target (0 = No escaló, 1 = Escaló):
Target
1    30
0    20
Name: count, dtype: int64

✓ UNIÓN COMPLETADA EXITOSAMENTE. Dimensiones finales: (50, 10)
Verificación de duplicados en la clave Ticker: 0

Vista previa del Dataset Unificado (Aún Sucio):
              Security             GICS Sector Ticker  Rendimiento_5Y  Target  \
0                   3M             Industrials    MMM        0.339498       0   
1          A. O. Smith             Industrials    AOS        0.357398       0   
2  Abbott Laboratories             Health Care    ABT        0.256830       0   
3               AbbVie             Health Care   ABBV        1.634203       1   
4            Accenture  Information Technology    ACN        0.123956       0   

  Revenue_Growth Net_Income Profit_Margin  Debt_to_Equity PE_Ratio  
0            NaN      (29M)           N/A            0.4

In [ ]:
import re

print("="*80)
print(" PASO 3: LIMPIEZA PESADA (REGEX, TRATAMIENTO DE INF Y IMPUTACIÓN POR SECTOR) ")
print("="*80)

# Copia de trabajo para preservar el DataFrame sucio original
df_clean = df_raw.copy()

# -----------------------------------------------------------------------------
# 3.1. FUNCIÓN PARSER CON REGEX PARA DATOS FINANCIEROS SUCIOS
# -----------------------------------------------------------------------------
def parse_financial_string(val):
    if pd.isna(val) or val in ["N/A", "MISSING", "nan", "None"]:
        return np.nan

    val_str = str(val).strip()

    # Manejo de valores infinitos
    if val_str in ["inf", "-inf", "np.inf"]:
        return np.nan

    try:
        # Detectar números negativos expresados entre paréntesis: (12M) -> -12M
        is_negative = False
        if val_str.startswith('(') and val_str.endswith(')'):
            is_negative = True
            val_str = val_str[1:-1]

        # Eliminar símbolos de moneda, porcentajes, comas y espacios
        clean_str = re.sub(r'[\$%\,\s]', '', val_str)

        # Detectar multiplicadores de magnitud (K, M, B)
        multiplier = 1.0
        if clean_str.endswith('B'):
            multiplier = 1e9
            clean_str = clean_str[:-1]
        elif clean_str.endswith('M'):
            multiplier = 1e6
            clean_str = clean_str[:-1]
        elif clean_str.endswith('K'):
            multiplier = 1e3
            clean_str = clean_str[:-1]

        num = float(clean_str) * multiplier
        return -num if is_negative else num
    except Exception:
        return np.nan

# -----------------------------------------------------------------------------
# 3.2. APLICAR PARSEO CON REGEX A LAS COLUMNAS NUMÉRICAS
# -----------------------------------------------------------------------------
cols_a_limpiar = ['Revenue_Growth', 'Net_Income', 'Profit_Margin', 'Debt_to_Equity', 'PE_Ratio']

print("\n--- Convirtiendo columnas contables de texto a float64 mediante Regex ---")
for col in cols_a_limpiar:
    df_clean[col] = df_clean[col].apply(parse_financial_string)

print("Tipos de datos tras la conversión:")
print(df_clean[cols_a_limpiar].dtypes)

# -----------------------------------------------------------------------------
# 3.3. IMPUTACIÓN DE NaN UTILIZANDO LA MEDIANA POR SECTOR GICS
# -----------------------------------------------------------------------------
print("\n--- Conteo de nulos antes de imputar por sector ---")
print(df_clean[cols_a_limpiar].isnull().sum())

# Imputar con la mediana según el sector industrial de Wikipedia
for col in cols_a_limpiar:
    df_clean[col] = df_clean.groupby('GICS Sector')[col].transform(lambda x: x.fillna(x.median()))
    # Cobertura en caso de que un sector completo no tenga datos
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("\n--- Conteo de nulos tras la imputación por mediana de sector ---")
print(df_clean[cols_a_limpiar].isnull().sum())

print("\n" + "="*80)
print("✓ LIMPIEZA PESADA COMPLETADA EXITOSAMENTE")
print("="*80)
print("\nVista previa del Dataset 100% Limpio y Numérico:")
print(df_clean[['Ticker', 'GICS Sector', 'Revenue_Growth', 'Net_Income', 'Profit_Margin', 'PE_Ratio', 'Target']].head(5))

 PASO 3: LIMPIEZA PESADA (REGEX, TRATAMIENTO DE INF Y IMPUTACIÓN POR SECTOR) 

--- Convirtiendo columnas contables de texto a float64 mediante Regex ---
Tipos de datos tras la conversión:
Revenue_Growth    float64
Net_Income        float64
Profit_Margin     float64
Debt_to_Equity    float64
PE_Ratio          float64
dtype: object

--- Conteo de nulos antes de imputar por sector ---
Revenue_Growth    10
Net_Income         0
Profit_Margin      9
Debt_to_Equity    15
PE_Ratio          21
dtype: int64

--- Conteo de nulos tras la imputación por mediana de sector ---
Revenue_Growth    0
Net_Income        0
Profit_Margin     0
Debt_to_Equity    0
PE_Ratio          0
dtype: int64

✓ LIMPIEZA PESADA COMPLETADA EXITOSAMENTE

Vista previa del Dataset 100% Limpio y Numérico:
  Ticker             GICS Sector  Revenue_Growth    Net_Income  Profit_Margin  \
0    MMM             Industrials    1.405100e+11 -2.900000e+07           9.84   
1    AOS             Industrials    1.125200e+11  6.300000e+10 

Graficos de corelacion
Que aprenda de las caracteristicas, no de algo exacto como el porcentaje